# Agent 3: Synthesizer

## Τι κάνει αυτός ο agent;

Ο **Synthesizer** παίρνει τα validated chunks από τον Retriever & Evaluator
και παράγει μια **συνεκτική απάντηση** σε φυσική γλώσσα.

Τρεις κανόνες που πρέπει να τηρεί:

1. **Χρησιμοποιεί ΜΟΝΟ** την πληροφορία από τα chunks — δεν επινοεί (no hallucination)
2. **Cite-άρει** κάθε ισχυρισμό: `[Paper: τίτλος, χρονιά]`
3. **Ομολογεί** αν η πληροφορία είναι ελλιπής — δεν κάνει educated guesses

## Γιατί είναι σημαντικό;

Ενα LLM χωρίς grounding θα επινοήσει απαντήσεις (hallucinate).
Με σαφές system prompt + retrieved context, **αναγκάζουμε** το μοντέλο
να στηριχτεί μόνο στα papers.


In [1]:
from openai import OpenAI

OLLAMA_BASE  = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2:3b"

llm = OpenAI(base_url=OLLAMA_BASE, api_key="ollama")

def call_llm(system_prompt: str, user_message: str, temperature: float = 0.3) -> str:
    response = llm.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content

## Το System Prompt του Synthesizer

Πρόσεξε ότι:
- Ζητάμε **inline citations** σε συγκεκριμένο format
- Ζητάμε ρητή δήλωση όταν η πληροφορία λείπει
- Χρησιμοποιούμε λίγο υψηλότερο `temperature=0.3` (vs 0.1) για πιο φυσική γλώσσα

In [2]:
SYNTHESIZER_PROMPT = """
You are an Academic Research Synthesizer for an AI/ML literature assistant.

You will receive a question and numbered context passages from academic papers.
Write a clear, accurate answer using ONLY the information in the provided passages.

Rules:
1. Cite sources inline: [Paper: <title>, <year>]
   Example: "Spiking networks use spike timing [Paper: SpikeGPT, 2023]."
2. If passages lack sufficient information, say explicitly:
   "Based on the available papers, I cannot fully answer this. [what is missing]"
3. NEVER invent information not present in the passages.
4. Write 2-4 clear paragraphs. Be concise and accurate.
5. End with a "Sources:" section listing all cited papers.
"""

def build_context(question: str, chunks: list[dict]) -> str:
    """Φτιάχνει το user message με numbered passages."""
    passages = []
    for i, c in enumerate(chunks, 1):
        # Εξάγουμε χρονιά από arxiv_id (τα 4 πρώτα ψηφία είναι YY.MM)
        arxiv_id  = c.get("arxiv_id", "unknown")
        year      = "20" + arxiv_id[:2] if len(arxiv_id) >= 4 and arxiv_id[:2].isdigit() else "?"
        title     = c.get("title", "Unknown")
        text      = c.get("text", "")
        passages.append(f"[{i}] {title} ({year})\n{text}")

    context = "\n\n".join(passages)
    return f"Question: {question}\n\nContext passages:\n{context}"

def synthesize(question: str, chunks: list[dict]) -> str:
    """Παράγει την τελική απάντηση."""
    if not chunks:
        return "Δεν βρέθηκε σχετική πληροφορία στη βάση δεδομένων."
    user_msg = build_context(question, chunks)
    return call_llm(SYNTHESIZER_PROMPT, user_msg, temperature=0.3)

print("synthesize() ready.")

synthesize() ready.


## Δοκιμή 1: Με καλά chunks

Χρησιμοποιούμε mock chunks για να δείξουμε την απόκριση ανεξάρτητα από το ChromaDB.

In [3]:
mock_chunks_good = [
    {
        "arxiv_id": "2302.13939",
        "title":    "SpikeGPT: Generative Pre-trained Language Model with Spiking Neural Networks",
        "authors":  "Rui-Jie Zhu et al.",
        "text":     "Spiking Neural Networks (SNNs) communicate via discrete spikes rather than continuous "
                    "activations, resulting in significantly lower energy consumption compared to traditional "
                    "Artificial Neural Networks (ANNs). Spike-based computation is inherently sparse: a neuron "
                    "only transmits a signal when it fires, which maps naturally to neuromorphic hardware.",
    },
    {
        "arxiv_id": "2401.17911",
        "title":    "SNNLP: Energy-Efficient Natural Language Processing Using Spiking Neural Networks",
        "authors":  "R. Alexander Knipper et al.",
        "text":     "Our experiments show that SNN-based models achieve up to 94% energy reduction compared "
                    "to equivalent transformer architectures on neuromorphic hardware. The trade-off is a "
                    "small accuracy gap of 2-5% on standard NLP benchmarks, which narrows as training "
                    "techniques improve.",
    },
    {
        "arxiv_id": "2406.03287",
        "title":    "SpikeLM: Towards General Spike-Driven Language Modeling",
        "authors":  "Xingrun Xing et al.",
        "text":     "We introduce an elastic bi-spiking mechanism that allows the model to adaptively "
                    "control spike frequency per layer, balancing accuracy and energy efficiency. "
                    "SpikeLM achieves competitive performance on language modeling tasks while maintaining "
                    "the biological plausibility of spike-based communication.",
    },
]

question = "What are the energy efficiency advantages of spiking neural networks compared to traditional neural networks?"

print(f"Q: {question}\n")
print("=" * 70)
answer = synthesize(question, mock_chunks_good)
print(answer)

Q: What are the energy efficiency advantages of spiking neural networks compared to traditional neural networks?

Spiking Neural Networks (SNNs) offer several energy efficiency advantages compared to traditional Artificial Neural Networks (ANNs). One key benefit is the inherent sparsity of spike-based computation, where a neuron only transmits a signal when it fires [Paper: SpikeGPT, 2023]. This sparse communication maps naturally to neuromorphic hardware, which can lead to significant reductions in energy consumption.

Studies have demonstrated that SNN-based models can achieve substantial energy savings. For example, experiments on neuromorphic hardware show that SNNs can reduce energy consumption by up to 94% compared to equivalent transformer architectures [Paper: SNNLP, 2024]. While there is a small accuracy gap of 2-5% on standard NLP benchmarks, this trade-off narrows as training techniques improve.

The design of the model also plays a crucial role in achieving energy efficienc

## Δοκιμή 2: Με ανεπαρκή chunks (πρέπει να ομολογήσει αδυναμία)

Ο synthesizer **δεν πρέπει να επινοήσει** αν τα chunks δεν έχουν την απάντηση.

In [4]:
mock_chunks_insufficient = [
    {
        "arxiv_id": "2302.13939",
        "title":    "SpikeGPT: Generative Pre-trained Language Model with Spiking Neural Networks",
        "authors":  "Rui-Jie Zhu et al.",
        "text":     "SpikeGPT is a language model that uses spiking neural networks for text generation. "
                    "It achieves competitive results on language modeling benchmarks.",
    },
]

question_hard = "How do quantum error correction codes compare to classical error correction in terms of qubit overhead?"

print(f"Q: {question_hard}\n")
print("=" * 70)
answer2 = synthesize(question_hard, mock_chunks_insufficient)
print(answer2)

Q: How do quantum error correction codes compare to classical error correction in terms of qubit overhead?

I must point out that the provided passage does not mention quantum error correction codes or classical error correction in terms of qubit overhead.

Based on the available papers, I cannot fully answer this question regarding the comparison between quantum error correction codes and classical error correction in terms of qubit overhead. The passage provided appears to be unrelated to the topic at hand.

If you could provide additional context or passages that discuss quantum error correction codes and their comparison to classical error correction, I would be happy to try and assist you further.

Sources:


## Δοκιμή 3: Με άδεια chunks

Αν ο Retriever δεν βρήκε τίποτα.

In [5]:
print(f"Q: {question_hard}\n")
print("Empty chunks  ")
print(synthesize(question_hard, []))

Q: How do quantum error correction codes compare to classical error correction in terms of qubit overhead?

Empty chunks  
Δεν βρέθηκε σχετική πληροφορία στη βάση δεδομένων.


## Δοκιμή 4: Με αληθινά chunks από ChromaDB

In [6]:
from pathlib import Path
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

CHROMA_PATH = Path("../RAG_demo/chroma_db")
COLLECTION  = "arxiv_papers"

embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client       = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection   = client.get_collection(name=COLLECTION, embedding_function=embedding_fn)

def retrieve(query: str, k: int = 4) -> list[dict]:
    results = collection.query(query_texts=[query], n_results=k)
    return [
        {"text": doc, "title": meta["title"], "arxiv_id": meta["arxiv_id"],
         "authors": meta["authors"], "score": round(1 - dist, 4)}
        for doc, meta, dist in zip(
            results["documents"][0], results["metadatas"][0], results["distances"][0]
        )
    ]

# End-to-end: retrieve  synthesize
real_question = "How do spiking neural networks handle natural language processing tasks?"
real_chunks   = retrieve(real_question)

print(f"Q: {real_question}")
print(f"Retrieved {len(real_chunks)} real chunks from ChromaDB\n")
print("=" * 70)
print(synthesize(real_question, real_chunks))

/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11521.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q: How do spiking neural networks handle natural language processing tasks?
Retrieved 4 real chunks from ChromaDB

Based on the provided context passages, it appears that spiking neural networks (SNNs) are not explicitly mentioned in the papers related to natural language processing (NLP). However, some papers discuss the use of transformers and other architectures that may be relevant to NLP tasks.

One passage from [1] discusses "syntactic islands" in transformer LMs, which could potentially relate to SNNs. The authors suggest that neural language models can be viewed as psycholinguistic subjects, but they do not explicitly discuss the use of SNNs for NLP tasks.

Another passage from [2] discusses the limitations of transformers and proposes a theoretical perspective on how to overcome them. While this paper does not mention SNNs specifically, it may provide insights into the design of more effective neural architectures for NLP tasks.

The papers [3], [4], and [5] discuss various as

## Ως LangGraph Node

In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class SynthesizerState(TypedDict):
    question: str
    chunks:   list[dict]
    answer:   str

def synthesizer_node(state: SynthesizerState) -> dict:
    """LangGraph node: παράγει την τελική απάντηση με citations."""
    answer = synthesize(state["question"], state["chunks"])
    print(f"[Synthesizer] Generated answer ({len(answer)} chars)")
    return {"answer": answer}

g = StateGraph(SynthesizerState)
g.add_node("synthesizer", synthesizer_node)
g.set_entry_point("synthesizer")
g.add_edge("synthesizer", END)

synth_app = g.compile()

result = synth_app.invoke({
    "question": real_question,
    "chunks":   real_chunks,
    "answer":   "",
})
print(result["answer"])

[Synthesizer] Generated answer (3074 chars)
Based on the provided context passages, it appears that spiking neural networks (SNNs) are not explicitly mentioned in the papers as handling natural language processing (NLP) tasks. However, some of the papers do discuss the use of transformer models and other NLP-related concepts.

The paper "Causal Drawbridges: Characterizing Gradient Blocking of Syntactic Islands in Transformer LMs" [1] discusses gradient blocking in transformer models, which can be a challenge for NLP tasks. The authors suggest that understanding this phenomenon is crucial for improving the performance of these models on various NLP tasks.

Another paper, "Barriers to Universal Reasoning With Transformers" [2], explores the limitations of transformers and proposes ways to overcome them. While not specifically focused on SNNs, the paper discusses the importance of considering the limitations of transformer models when designing new architectures.

The papers "FABLE: Fine-

## Τι μάθαμε

1. **System prompt** ελέγχει το behavior: cite, no hallucination, admit ignorance
2. **`temperature=0.3`** (λίγο υψηλότερο) για πιο φυσική γλώσσα στη σύνθεση
3. **Πάντα handle empty chunks** πριν καλέσετε το LLM
4. **Format citations ομοιόμορφα** — ο χρήστης μπορεί να τα ελέγξει

---

Στο τελευταίο notebook (`05_full_agentic_rag.ipynb`) συνδέουμε και τους τρεις agents
σε ένα ολοκληρωμένο LangGraph pipeline.